# 📈 Getting Constant Maturity Yields From FRED
<br>

<div style="display: flex; flex-wrap: wrap; align-items: center; gap: 15px; margin-bottom: 25px; padding-bottom: 15px; border-bottom: 1px solid #eaeaea;">
  
  <a href="https://colab.research.google.com/github/PatrickJHess/Volume-Four-Chapter-One/blob/master/colab/Colab_Getting_Constant_Maturity_Yields_From_FRED.ipynb" target="_blank" style="display: flex; align-items: center;">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="height: 28px; margin: 0;">
  </a>

  <a href="https://mybinder.org/v2/gh/PatrickJHess/Volume-Four-Chapter-One/master?urlpath=lab/tree/notebooks/Getting_Constant_Maturity_Yields_From_FRED.ipynb" target="_blank" style="background-color: #f5a252; color: white; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">🚀</span> Launch Live in Binder
  </a>

  <a href="https://patrickjhess.github.io/Volume-Four-Chapter-One/" style="background-color: #f1f3f4; color: #3c4043; border: 1px solid #dadce0; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">⬅️</span> Return to Main Book
  </a>
</div>

This notebook puts our secure key management and data pipeline to FRED to work. Before we dive in, you need to have some familiarity with FRED series IDs and how to extract them. As you will see, the `get_series` method of `FredReader` deftly handles incorrect series IDs and can be used to sort out the correct ones.

Start off by watching this short video on the FRED database and its series IDs


[![Getting Series IDs From Fred](https://img.youtube.com/vi/ZP1hKFRZAz4/0.jpg)](https://youtu.be/ZP1hKFRZAz4)

<details style="border: 1px solid #b8daff; border-radius: 8px; padding: 15px; background-color: #f8fbff; margin: 20px 0;">
  <summary style="cursor: pointer; font-weight: bold; font-size: 1.1em; color: #004085; font-family: sans-serif; list-style-position: inside;">
    <span style="margin-right: 8px;">🛠️</span> Notebook Setup: Why the "Try/Except" Imports?
  </summary>
  <div style="margin-top: 15px; padding-top: 15px; border-top: 1px solid #b8daff; color: #333333; font-family: sans-serif; font-size: 0.95em;">
    <b>The Goal:</b><br>
    To ensure this notebook runs perfectly whether you are using <b>Google Colab</b>, a local <b>Jupyter instance</b>, or a remote server without you having to manually install software.<br><br>
    <b>Key Concepts in this Section:</b>
    <ul style="line-height: 1.6; margin-bottom: 0;">
      <li><b>Standard Libraries:</b> Modules like <code>os</code>, <code>sys</code>, and <code>datetime</code> come "in the box" with Python. We use them for system tasks and date math.</li>
      <li><b>External Libraries:</b> NumPy and Pandas are the "heavy hitters" for data. They aren't always installed by default.</li>
      <li><b>The <code>try/except</code> Logic:</b> This is a safety net.
        <ol style="margin-top: 4px; margin-bottom: 4px; padding-left: 20px;">
          <li>We <b>try</b> to import the library.</li>
          <li>If it fails (because it's not installed), the <b>except</b> block triggers a <code>!pip install</code> to download it automatically.</li>
        </ol>
      </li>
      <li><b>Aliasing (<code>as np</code>):</b> We rename <code>numpy</code> to <code>np</code> to save keystrokes. In professional finance code, <code>np</code> and <code>pd</code> are the universal shorthand.</li>
    </ul>
  </div>
</details>

## Importing libraries, modules, And functions

Modules that are included in the standard Python library are imported. When necessary, other modules or libraries are installed before they are imported. (see [Control Statements](https://patrickjhess.github.io/Introduction-To-Python-For-Financial-Python/Control_Statements.html#the-try-and-except)).

```
try:
    import pandas as pd
except:
    !pip -q install pandas
    import pandas as pd
```

In [ ]:
import os
import sys
import requests
from datetime import date, datetime
from types import ModuleType

try:
    import pandas as pd
except:
    !pip -q install pandas
    import pandas as pd

<details>
<summary><b style="font-size:1.2em; color: #1976d2; cursor: pointer;">👍 Cloud-Loading: How In-Memory Modules Work</b></summary>
<br>
<p><b>The Logic:</b><br>
Usually, Python looks for modules as <code>.py</code> files on your hard drive. Here, we are "tricking" Python into treating a string of text from a URL as a live library.</p>

<p><b>The Workflow:</b></p>
<ol>
<li><b>Fetch:</b> <code>requests.get(url)</code> grabs the raw text of your Python script from Dropbox.</li>
<li><b>Instantiate:</b> <code>ModuleType(module_name)</code> creates an empty "container" in your computer's RAM.</li>
<li><b>Execute:</b> <code>exec(code, module.__dict__)</code> runs that text inside the container, turning text into live functions.</li>
<li><b>Register:</b> By adding it to <code>sys.modules</code>, we tell Python: <em>"If I try to import this later, don't look on the disk—look right here in the memory."</em></li>
</ol>

<p><b>Why do this?</b><br>
It makes your notebooks <b>100% portable</b>. A user can open this in a brand-new environment, and as long as they have an internet connection, all your custom financial functions will "just work."</p>
</details>

In [ ]:
# Define the URL of the Python module to be downloaded from Dropbox.
# The 'dl=1' parameter in the URL forces a direct download of the file content.
url= 'https://www.dropbox.com/scl/fi/4y5hjxlfphh1ngvbgo77q/\
module_-basic_concepts_fixed_income.py?rlkey=6oxi7mgka42veaat79hcv8boz&st=87sztshr&dl=1'
module_name='basic_concepts_fixed_income'
# Send an HTTP GET request to the URL and store the server's response.
try:
    response = requests.get(url)
    module = ModuleType(module_name)
    exec(response.content.decode('utf-8'), module.__dict__)
    sys.modules[module_name] = module
    # Now we can import from our in-memory module
    from basic_concepts_fixed_income import (secure_key_setup)
    # Assign the FredReader attribute
    FredReader = module.FredReader
except requests.exceptions.RequestException as e:
    print(f"❌ Error: Could not fetch module from URL. {e}")
except Exception as e:
    print(f"❌ Error: Failed to execute or import the module. {e}")
    # Now we can import from our in-memory module)

## ✅ Authenticate your FRED API key

In [ ]:
secure_key_setup("fred_key")

## 🏦 🆔 Series IDs for constant maturity yields and the secured overnight funding rate SOFR

Generate a potential list of Series IDs for constant maturity yields as `f` strings..

*   **Monthly Series**: all monthly maturities between one and eleven months

```
monthly_ids = [f"DGS{i}MO" for i in range(1, 12)]
```


*   **Yearly Series**: all years between one and thirty years

```
yearly_ids = [f"DGS{i}" for i in range(1, 31)]
```

Secured overnight Id is 'SOFR'.  The list of all IDs is series_id.

In [ ]:
# Generate months 1-11 and years 1-30 programmatically
monthly_ids = [f"DGS{i}MO" for i in range(1, 12)]
yearly_ids = [f"DGS{i}" for i in range(1, 31)]

# Combine everything together with SOFR
series_ids = ['sofr'] + monthly_ids + yearly_ids

## 🔗  Accessing constant maturity yields for May 2026 with `FredReader`.

The `get_series` method of `FredReader`

In [ ]:
# create an instance of FredReader
fred_data=FredReader()

# call the method for the class
yield_data=fred_data.get_series(series_ids,start_date='2026-05-01')
display(yield_data)

 ### ✍️ FRED Data Challenge
>


*   Access Overnight Secured Funding Rate And Par Yield For Thirty Year Maturity between January 1, 2026 and May 20, 2026.
*   Did you access series from cache or FRED?


💡 Tip: Use the first and last column heads of `yield_data` as series IDs.
---


 <details>
 <summary><b>✅ Example Of Solution</b></summary>
 <br>

 **Example of Code**
 ```python
 # assuming imports of this notebook
 fred_data=FredReader()
 series_ids=[yield_data.columns[0],yield_data.columns[-1]]
 fred_data.get_series(series_ids,start_date='2026-01-01',end_date='2026-05-20')
 ```
 </details>

>


---
<details>
<summary style="cursor: pointer; color: #2196f3; font-weight: bold;">👉 Click here to reveal the answers</summary>
<div style="margin-top: 10px; padding: 10px; border-left: 3px solid #2196f3; background-color: #f9f9f9;">
The previous request for FRED data specified May 1, 2026 as the start date.  The data request can not be completed with cache.  The cache for the two series `sofr` and `DGS30` is updated.